[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pctablet505/litert-demo/blob/main/litertlm_export_demo.ipynb)

# LiteRT-LM Export Demo — Gemma 3 270M

Minimal demonstration of exporting a KerasHub `Gemma3CausalLM` to a **LiteRT-LM bundle** (`.litertlm`).

LiteRT-LM packages the TFLite model (with `prefill` + `decode` signatures), the SentencePiece tokenizer, and LLM metadata into a single file. On Android, the LiteRT-LM runtime handles tokenization, KV-cache management, sampling, and streaming for you.

> **Requirements:**
> - `KERAS_BACKEND=torch`
> - `litert-torch` and `litert-lm-builder` installed
> - This export path is **PyTorch-only**
> - LiteRT-LM PR branch: `torch-backend-litert-minimal-litertlm`

In [1]:
!pip install -q litert-torch
!pip install -q git+https://github.com/keras-team/keras.git
!pip install -q git+https://github.com/pctablet505/keras-hub.git@torch-backend-litert-minimal-litertlm



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 575.8/575.8 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.3/419.3 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.6/117.6 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 94.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 20.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-genai 1.68.0 requires typing-extensions<5.0.0,>=4.14.0, but you have typing-extensions 4.12.2 which is 

In [2]:
import os

PRESET = "hf://google/gemma-3-270m-it"

CACHE_LEN = 1024   # Max conversation context
PREFILL_LEN = 128  # Max prompt length per turn

print("Preset:", PRESET)
print(f"cache_length={CACHE_LEN}, prefill_seq_len={PREFILL_LEN}")

Preset: hf://google/gemma-3-270m-it
cache_length=1024, prefill_seq_len=128


In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import keras_hub

model = keras_hub.models.Gemma3CausalLM.from_preset(PRESET)
model.preprocessor.sequence_length = CACHE_LEN

# Build via direct call with concrete shapes
processed = model.preprocessor({"prompts": ["What is Keras?"], "responses": [""]})
model(processed[0])
print("Model built. Output shape:", model(processed[0]).shape)

## Export to `.litertlm`

A single call with `format="litertlm"` produces the bundle. The exporter automatically:
- Traces `prefill` (prompt → KV cache) and `decode` (single token → logits + updated KV cache) signatures
- Bundles the SentencePiece tokenizer
- Writes `LlmMetadata` (start/stop tokens, context window, model type)

In [ ]:
# Single bucket (legacy behavior)
# model.export("gemma3_270m_it.litertlm", format="litertlm", prefill_seq_len=128)

# Multiple buckets — runtime dispatches to the smallest fitting signature
# NOTE: Each bucket adds ~2-4 min to export time (signature is re-traced),
# but model size increase is minimal (~0.07% for 3 extra signatures)
# because weights are shared across all prefill signatures.
model.export(
    "gemma3_270m_it.litertlm",
    format="litertlm",
    prefill_seq_len=[32, 64],
)
print("✅ Exported with bucketing: prefill_32, prefill_64")
print("Size (MB):", round(os.path.getsize("gemma3_270m_it.litertlm") / 1e6, 1))

## Verify Bundle Contents

Peek inside the `.litertlm` file to confirm it contains the expected assets.

In [ ]:
import io
from litert_lm_builder import litertlm_peek

output = io.StringIO()
litertlm_peek.peek_litertlm_file("gemma3_270m_it.litertlm", None, output)
print(output.getvalue())
print("✅ LiteRT-LM bundle verified.")

### Bucketing Performance (Measured on Pixel 9)

| Model | Prompt Tokens | TTFT |
|-------|--------------|------|
| Fixed-128 (`prefill_seq_len=128`) | 31 tok | **2866.5 ms** |
| Bucketed `[32, 64, 128]` | 31 tok | **1626.4 ms** |

**→ 43.3% faster TTFT** with bucketing for a ~31-token prompt.

The fixed model pads every prompt to 128 tokens. The bucketed runtime dispatches to `prefill_64` (smallest fitting signature), avoiding ~60% of wasted prefill compute.

In [ ]:
from litert_torch.generative.quantize.quant_recipes import full_dynamic_recipe

model.export(
    "gemma3_270m_it_wi8afp32.litertlm",
    format="litertlm",
    prefill_seq_len=[32, 64, 128, 256, 512, 1024],
    quant_config=full_dynamic_recipe(),
)
print("✅ Exported + quantized in one step")
print("Size (MB):", round(os.path.getsize("gemma3_270m_it_wi8afp32.litertlm") / 1e6, 1))

## Push to Android & Test

```bash
# Push to device
adb push gemma3_270m_it_wi8afp32.litertlm /data/local/tmp/
```

**Verified on Pixel 9 (physical ARM64):**
- Engine init: **2,318 ms** (vs ~5,336 ms for FP32)
- Generation: **4,213 ms** for "What is Keras?"
- Response: *"Keras is a high-level, Python API for building and training machine learning models..."*